# E2 / E3 — Transient behaviour under bursts

Questions (`IMPLEMENTATION_PLAN.md` §14.1):
* **E2 step burst** (0.3 → 1.5 → 0.3 ρ, Main Flow §8): how fast does the thermostat cut B, does p95 stay near T_SLO, and how long does recovery take?
* **E3 MMPP-2** (random on/off bursts): same questions under realistic traffic.
* **Who absorbs the cuts?** Per-group ef_used / ef_pref for `laef_wf` vs `laef_uniform`.

**Inputs** (plan §11.3), per run directory `results/<experiment>/<run_id>/`:
* `config.json` — resolved config (absolute rates in `workload`, `t_slo_s`);
* `timeseries.csv` — one row per sensor window (`t_s, B, B_max, B_min, D, p95_ms, s_p95_ms, Q, W_hat_ms, u_ns, ES_ms, lambda_hat, w_target, water_level, uniform_fraction, zone, guard_fired, floor_mode, slo_violated`);
* `queries.csv` — one row per query (`qid, eval_idx, warmup, t_arrival_us, wait_us, service_us, latency_us, status, score, group, ef_pref, ef_used, forced_floor, floor_mode, ndis, probe_ndis, planned_work, recall`);
* `summary.json` (incl. `recovery_time_s` for step profiles).

In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'configs').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
FIG = ROOT / 'results' / 'figures' / 'e2_e3_bursts'
FIG.mkdir(parents=True, exist_ok=True)

COLORS = {'static_ef': 'tab:gray', 'ada_ef': 'tab:blue', 'laef_uniform': 'tab:orange', 'laef_wf': 'tab:green'}
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

def load_run(run_dir):
    run_dir = pathlib.Path(run_dir)
    r = {'dir': run_dir, 'id': run_dir.name,
         'config': json.loads((run_dir / 'config.json').read_text()),
         'summary': json.loads((run_dir / 'summary.json').read_text()),
         'ts': pd.read_csv(run_dir / 'timeseries.csv'),
         'q': pd.read_csv(run_dir / 'queries.csv')}
    r['policy'] = r['config'].get('policy') or 'unknown'
    r['dataset'] = r['config'].get('dataset', '?')
    r['t_slo_ms'] = 1e3 * r['config'].get('t_slo_s', r['summary'].get('slo_ms', np.nan) / 1e3)
    return r

def list_runs(experiment):
    d = RESULTS / experiment
    return sorted(p for p in d.iterdir() if (p / 'summary.json').exists()) if d.exists() else []

def phase_spans(cfg):
    # (start_s, end_s, qps) of each step phase; empty for non-step profiles
    w = cfg.get('workload', {})
    if w.get('profile') != 'step':
        return []
    spans, t = [], 0.0
    for ph in w.get('phases', []):
        spans.append((t, t + ph['duration_s'], ph.get('qps', 0)))
        t += ph['duration_s']
    return spans

print({e: len(list_runs(e)) for e in ['e2_step_burst', 'e3_mmpp']})

## 1. Time series of one run per policy

Shaded areas are the high-rate step phases. Panels: offered load, budget B (with B_max / B_min), window p95 vs T_SLO, queue depth Q, water level λ* and floor mode. `REPEAT` selects which repeat to show.

In [ ]:
REPEAT = 0

def pick_runs(experiment, repeat=REPEAT):
    out = {}
    for p in list_runs(experiment):
        r = load_run(p)
        if int(r['config'].get('repeat', r['summary'].get('repeat', 0))) != repeat:
            continue
        out.setdefault(r['dataset'], {})[r['policy']] = r
    return out

def shade(ax, spans):
    if not spans:
        return
    peak = max(s[2] for s in spans)
    for a, b, qps in spans:
        if qps == peak:
            ax.axvspan(a, b, color='red', alpha=0.08)

def plot_timeseries(runs_by_policy, title, fname):
    fig, ax = plt.subplots(6, 1, figsize=(11, 13), sharex=True)
    spans = []
    for pol, r in runs_by_policy.items():
        ts, q, c = r['ts'], r['q'], COLORS.get(pol, 'k')
        spans = phase_spans(r['config']) or spans
        arr = q.t_arrival_us / 1e6
        bins = np.arange(0, arr.max() + 0.1, 0.1)
        cnt, edges = np.histogram(arr, bins=bins)
        ax[0].plot(edges[:-1], cnt / 0.1, color=c, lw=0.8, label=pol)
        if ts.B.notna().any() and pol.startswith('laef'):
            ax[1].plot(ts.t_s, ts.B / 1e6, color=c, label=f'{pol} B')
            ax[1].plot(ts.t_s, ts.B_max / 1e6, color=c, ls=':', lw=0.8)
            ax[1].plot(ts.t_s, ts.B_min / 1e6, color=c, ls='--', lw=0.8)
            ax[4].plot(ts.t_s, ts.water_level, color=c)
            ax[5].step(ts.t_s, ts.floor_mode.astype(int), where='post', color=c)
        ax[2].plot(ts.t_s, ts.p95_ms, color=c)
        ax[3].plot(ts.t_s, ts.Q, color=c)
        ax[2].axhline(r['t_slo_ms'], color='k', ls='--', lw=1)
    for a in ax:
        shade(a, spans)
    ax[0].set_ylabel('arrivals / s'); ax[0].legend(fontsize=7, ncol=4)
    ax[1].set_ylabel('B [M dc/s]\n(: B_max, -- B_min)')
    ax[2].set_ylabel('window p95 [ms]'); ax[2].set_yscale('log')
    ax[3].set_ylabel('queue depth Q')
    ax[4].set_ylabel('water level λ*'); ax[4].set_yscale('symlog', linthresh=1e-7)
    ax[5].set_ylabel('floor mode'); ax[5].set_xlabel('time [s]')
    fig.suptitle(title, y=1.0)
    fig.tight_layout()
    fig.savefig(FIG / fname, bbox_inches='tight')
    plt.show()

for exp in ['e2_step_burst', 'e3_mmpp']:
    for ds, by_pol in pick_runs(exp).items():
        plot_timeseries(by_pol, f'{exp}: {ds} (repeat {REPEAT})', f'{exp}_{ds}_timeseries.png')

## 2. Latency and recall over time (per-query, rolling)

Rolling over arrival time; makes the transient after the burst start/end visible at query granularity.

In [ ]:
ROLL = 200  # queries
for exp in ['e2_step_burst', 'e3_mmpp']:
    for ds, by_pol in pick_runs(exp).items():
        fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
        spans = []
        for pol, r in by_pol.items():
            q = r['q'][~r['q'].warmup.astype(bool)].sort_values('t_arrival_us')
            t = q.t_arrival_us / 1e6
            ax[0].plot(t, (q.latency_us / 1e3).rolling(ROLL).quantile(0.95), color=COLORS.get(pol, 'k'), label=pol, lw=0.9)
            ax[1].plot(t, q.recall.rolling(ROLL).mean(), color=COLORS.get(pol, 'k'), lw=0.9)
            spans = phase_spans(r['config']) or spans
            ax[0].axhline(r['t_slo_ms'], color='k', ls='--', lw=1)
        for a in ax:
            shade(a, spans)
        ax[0].set(xlabel='arrival time [s]', ylabel=f'rolling p95 latency [ms] ({ROLL} q)', yscale='log', title=f'{exp}: {ds}')
        ax[1].set(xlabel='arrival time [s]', ylabel=f'rolling mean recall ({ROLL} q)')
        ax[0].legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(FIG / f'{exp}_{ds}_rolling.png')
        plt.show()

## 3. Who absorbs the cuts?

During the high-rate phase (E2) or the whole run (E3): distribution of `ef_used / ef_pref` per Ada-ef difficulty group. Water-filling should cut easy groups first and leave hard groups closer to 1; uniform cut shrinks every group by a similar fraction.

In [ ]:
def burst_queries(r):
    q = r['q'][(~r['q'].warmup.astype(bool)) & (r['q'].status == 'ok') & (r['q'].group >= 0)].copy()
    spans = phase_spans(r['config'])
    if spans:
        peak = max(s[2] for s in spans)
        a, b = next((s[0], s[1]) for s in spans if s[2] == peak)
        t = q.t_arrival_us / 1e6
        q = q[(t >= a) & (t < b)]
    q['ratio'] = q.ef_used / q.ef_pref.clip(lower=1)
    return q

for exp in ['e2_step_burst', 'e3_mmpp']:
    for ds, by_pol in pick_runs(exp).items():
        pols = [p for p in ['laef_uniform', 'laef_wf'] if p in by_pol]
        if not pols:
            continue
        fig, ax = plt.subplots(1, len(pols) + 1, figsize=(5 * (len(pols) + 1), 3.6))
        ax = np.atleast_1d(ax)
        rows = []
        for i, pol in enumerate(pols):
            q = burst_queries(by_pol[pol])
            groups = sorted(q.group.unique())
            ax[i].boxplot([q[q.group == g].ratio for g in groups], showfliers=False)
            ax[i].set_xticks(range(1, len(groups) + 1), [f'g{g}' for g in groups])
            ax[i].set(ylim=(0, 1.05), ylabel='ef_used / ef_pref', title=f'{pol}')
            for g in groups:
                s = q[q.group == g]
                rows.append({'policy': pol, 'group': g, 'n': len(s), 'mean ef ratio': s.ratio.mean(),
                             'mean recall': s.recall.mean(), 'forced floor': s.forced_floor.mean()})
        t = pd.DataFrame(rows)
        for pol, s in t.groupby('policy'):
            ax[-1].plot(s.group, s['mean recall'], '-o', color=COLORS[pol], label=pol)
        ax[-1].set(xlabel='difficulty group', ylabel='mean recall in burst', title='recall per group')
        ax[-1].legend(fontsize=7)
        fig.suptitle(f'{exp}: {ds}', y=1.02)
        fig.tight_layout()
        fig.savefig(FIG / f'{exp}_{ds}_cuts_by_group.png', bbox_inches='tight')
        plt.show()
        display(t.round(3))

## 4. Summary over repeats

`recovery_time_s` = time from the end of the burst until B ≥ 0.95·B_max (step profile only).

In [ ]:
KEYS = ['mean_recall', 'p5_recall', 'p1_recall', 'p95_ms', 'slo_ms', 'slo_attainment', 'latency_excess_ms',
        'recovery_time_s', 'floor_mode_windows', 'guard_fires', 'forced_floor_rate']
tables = []
for exp in ['e2_step_burst', 'e3_mmpp']:
    for p in list_runs(exp):
        s = json.loads((p / 'summary.json').read_text())
        c = json.loads((p / 'config.json').read_text())
        tables.append({'experiment': exp, 'dataset': c.get('dataset'), 'policy': c.get('policy'),
                       **{k: s.get(k, np.nan) for k in KEYS}})
if tables:
    df = pd.DataFrame(tables)
    out = df.groupby(['experiment', 'dataset', 'policy']).agg(['mean', 'std'])
    out.to_csv(FIG / 'e2_e3_summary.csv')
    display(out.round(3))
else:
    print('no E2/E3 runs found')